# 🎬 AI Cloud Documentary & 2D Video Production Engine
**Up to 25 Minutes | 2D Animation & Documentary | Hindi Devanagari Shaping | Word-Synced Subtitles**

Is notebook ke zariye aap bina mobile ka processor ya storage use kiye Google Colab par 25 minute tak ki Full HD / 4K documentary aur 2D explainer video bana sakte hain.


In [ ]:
#@title 🛠️ Step 1: System Setup & Dependencies
# Click ▶️ Play button to install dependencies & Hindi Devanagari shaping fonts
import os
import sys

print('📦 Installing FFmpeg, Devanagari fonts, and HarfBuzz/Raqm shaping libs...')
os.system('apt-get update -qq && apt-get install -y -qq ffmpeg fonts-dejavu-core fonts-noto-core fonts-noto-sans-devanagari libfreetype6-dev libharfbuzz-dev libfribidi-dev libraqm-dev && fc-cache -fv > /dev/null')

repo_dir = '/content/ai-cloud-video-engine'
if not os.path.exists(repo_dir):
    print('📥 Cloning repository from GitHub...')
    os.system(f'git clone https://github.com/ankitputra160-jpg/ai-cloud-video-engine.git {repo_dir}')
else:
    print('🔄 Updating repository to latest commit...')
    os.system(f'cd {repo_dir} && git pull origin main')

os.chdir(repo_dir)
sys.path.insert(0, repo_dir)
print('📦 Installing Python requirements...')
os.system(f'pip install -q -r {repo_dir}/requirements.txt')
print('✅ Environment ready (up to 25-min 1080p/4K + 2D Animation + Hindi ASS Karaoke)!')


In [ ]:
#@title ☁️ Step 2: Mount Google Drive (Cloud Storage)
# Video sidhe Google Drive me save karne ke liye connect karein
from google.colab import drive
import os

drive.mount('/content/drive')
cloud_storage_dir = '/content/drive/MyDrive/AI_Documentaries'
os.makedirs(cloud_storage_dir, exist_ok=True)
print(f'✅ Google Drive Connected! Destination: {cloud_storage_dir}')


In [ ]:
#@title 🎥 Step 3: Enter Details & Generate Documentary (Up to 25 Min) { run: "auto" }
# Yahan topic, language, duration (10s - 1500s / 25 min), aur 2D/Documentary style chunein

Topic = "The Secrets of Ancient Indian Astronomy" #@param {type:"string"}
Language = "hi" #@param ["hi", "en"]
Duration = 60 #@param {type:"slider", min:10, max:1500, step:10}
Resolution = "1080p" #@param ["1080p", "720p", "4k"]
Style = "2D Animation" #@param ["Documentary", "2D Animation", "Vox Explainer", "Historical", "Scientific", "Biographical"]
Subtitle_Style = "ass_karaoke" #@param ["ass_karaoke", "pill_box", "srt_classic"]

import re
import os
import sys
import time
import subprocess
from IPython.display import Video, display

repo_dir = '/content/ai-cloud-video-engine'
if not os.path.exists(repo_dir):
    print('📥 Repository not found locally. Cloning now...')
    os.system('apt-get update -qq && apt-get install -y -qq ffmpeg fonts-dejavu-core fonts-noto-core fonts-noto-sans-devanagari libfreetype6-dev libharfbuzz-dev libfribidi-dev libraqm-dev && fc-cache -fv > /dev/null')
    os.system(f'git clone https://github.com/ankitputra160-jpg/ai-cloud-video-engine.git {repo_dir}')
    os.system(f'pip install -q -r {repo_dir}/requirements.txt')
else:
    os.system(f'cd {repo_dir} && git pull origin main -q')

os.chdir(repo_dir)
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

safe_name = re.sub(r'[^a-zA-Z0-9_]', '_', Topic)[:30].strip('_')
timestamp = int(time.time())
output_filename = f'{safe_name}_{Language}_{timestamp}.mp4'

# Determine destination folder (Google Drive preferred)
if os.path.exists('/content/drive/MyDrive'):
    cloud_storage_dir = '/content/drive/MyDrive/AI_Documentaries'
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        cloud_storage_dir = '/content/drive/MyDrive/AI_Documentaries'
    except Exception:
        cloud_storage_dir = '/content/output'

os.makedirs(cloud_storage_dir, exist_ok=True)
final_output_path = os.path.join(cloud_storage_dir, output_filename)
workspace_dir = f'/content/workspace_{timestamp}'

print('=' * 60)
print(f'🚀 Starting Production: {Topic} ({Style})')
print(f'🌐 Language: {Language} | ⏱️ Duration: {Duration}s | 🖥️ Resolution: {Resolution} | 💬 Subtitles: {Subtitle_Style}')
print(f'📁 Cloud Storage Destination: {final_output_path}')
print('=' * 60)

cmd = [
    sys.executable, '-m', 'engine.pipeline',
    '--topic', str(Topic),
    '--language', str(Language),
    '--duration', str(Duration),
    '--resolution', str(Resolution),
    '--style', str(Style),
    '--subtitle-style', str(Subtitle_Style),
    '--output', str(final_output_path),
    '--work-dir', str(workspace_dir)
]

env = os.environ.copy()
env['PYTHONPATH'] = repo_dir

process = subprocess.Popen(
    cmd,
    cwd=repo_dir,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()

if process.returncode == 0 and os.path.exists(final_output_path) and os.path.getsize(final_output_path) > 10000:
    size_mb = os.path.getsize(final_output_path) / (1024 * 1024)
    print('\n' + '=' * 60)
    print(f'🏆 VIDEO READY! File Size: {size_mb:.2f} MB')
    print(f'📁 Saved to: {final_output_path}')
    print('=' * 60)
    display(Video(final_output_path, embed=True, width=720))
else:
    print(f'\n❌ Video generation failed (Exit code: {process.returncode}).')
